# Phase 7 — Aligning a Judge to Human Judgement

Databricks AI Evals Tutorial | Phase 7 of 10 *(bonus track)*

Six phases of numbers, and not one of them has been validated.

Every score in this track came from an LLM judge. Those judges were written by the same
person who wrote the agent, the dataset, and the guidelines, and their verdicts have been
taken at face value throughout. The OpenAI evaluation guide names this precisely —
**"neglecting human feedback for metric validation"** — and prescribes calibrating automated
metrics against human judgement.

This phase does that: collect expert ratings, measure how far the judge is from them, and
use MemAlign to close the gap.

> **Requires a Databricks workspace** (labeling sessions and the Review App), plus a human
> willing to rate ~20 responses. That human can be you.

## The metric people reach for is the wrong one

The instinct, when asking "is my judge any good?", is to look at its **average score**.

That measures generosity, not quality. A judge that returns 5/5 for everything has a
superb average and is worthless. What matters is whether the judge **agrees with the people
whose standards it is supposed to encode**.

Once you measure agreement instead of average, the headline result of this phase stops
being surprising:

> **Alignment usually makes the judge's average score go *down*, and that is the success
> condition.** The unaligned judge was grading against generic best practice and inflating.
> The aligned one applies your experts' stricter standard. The agent did not get worse —
> the measurement got honest.

`GOTCHAS.md` flags this explicitly, because teams see the drop and conclude their agent
regressed.

## Three ways to measure agreement, and why you need more than one

| Measure | What it does | When it misleads |
|---|---|---|
| **Exact agreement** | fraction of identical ratings | Flatters badly on skewed distributions — two raters who both say "5" most of the time agree often by luck |
| **Cohen's kappa** | agreement corrected for chance | Treats every disagreement as total: 4-vs-5 counts the same as 1-vs-5 |
| **Quadratic weighted kappa** | penalises disagreement by squared distance | The right default for an ordinal scale; needs variance in the ratings to be meaningful |

That middle row is not a technicality. Below, two judges have **identical exact agreement
and identical plain kappa**, while one is systematically one point generous and the other is
essentially random.

In [ ]:
# ============ WHY THE CHOICE OF MEASURE MATTERS ============
import alignment as A

human        = [1, 2, 3, 4, 5, 2, 3, 4, 5, 1]
off_by_one   = [2, 3, 4, 5, 4, 3, 4, 5, 4, 2]   # consistently one point generous
uncorrelated = [5, 4, 1, 2, 1, 5, 1, 2, 1, 5]   # essentially random

print(f"{'JUDGE':<16}{'EXACT':>8}{'KAPPA':>9}{'WEIGHTED KAPPA':>17}")
print("-" * 50)
for label, judge in [("off-by-one", off_by_one), ("uncorrelated", uncorrelated)]:
    print(
        f"{label:<16}"
        f"{A.exact_agreement(judge, human):>8.2f}"
        f"{A.cohens_kappa(judge, human):>9.3f}"
        f"{A.quadratic_weighted_kappa(judge, human, 1, 5):>17.3f}"
    )

print()
print("Identical exact agreement. Identical plain kappa. Weighted kappa separates them")
print("decisively -- and the off-by-one judge is trivially fixable while the other is not.")


## Step 1 — Setup and a judge worth aligning

Alignment needs a judge whose criteria are genuinely domain-specific. `Safety()` is not a
candidate — "is this harmful" doesn't vary by company. Something like "did this response
meet our support quality bar" does, and that's exactly the kind of judgement generic best
practice gets wrong.

Note `feedback_value_type=float`: a 1-5 Likert gives experts room to express degree.
MemAlign is agnostic here — boolean and categorical work too — but a scale carries more
signal per label, which matters when labels are the expensive input.

In [ ]:
# ============ SETUP ============
import os

import mlflow

CATALOG_NAME = "<YOUR_CATALOG>"
SCHEMA_NAME = "<YOUR_SCHEMA>"
SQL_WAREHOUSE_ID = "<YOUR_SQL_WAREHOUSE_ID>"
SME_USERS = ["<you@example.com>"]          # who will do the rating

placeholders = [v for v in (CATALOG_NAME, SCHEMA_NAME, SQL_WAREHOUSE_ID, SME_USERS[0])
                if v.startswith("<")]
if placeholders:
    raise ValueError(f"Fill in your workspace values first. Still unset: {placeholders}")

mlflow.set_tracking_uri("databricks")
os.environ["TELCOASSIST_PROVIDER"] = "databricks"
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = SQL_WAREHOUSE_ID

EXPERIMENT = "/Shared/telcoassist-alignment"
experiment = mlflow.set_experiment(EXPERIMENT)
EXPERIMENT_ID = experiment.experiment_id
mlflow.langchain.autolog()

import agent
import traffic as T

# ONE name, used for the judge AND the label schema. See the warning below.
JUDGE_NAME = "support_quality"
DATASET_NAME = "telcoassist_alignment_set"

print(f"experiment : {EXPERIMENT} (id {EXPERIMENT_ID})")
print(f"judge name : {JUDGE_NAME}")


> ### The single most common way this fails
>
> **The label schema `name` must exactly match the judge `name`.** That is how `align()`
> pairs each expert rating with the judge score on the same trace. If they differ,
> alignment doesn't error — it finds no pairs, learns nothing, and returns a judge
> identical to the one you started with.
>
> That's why `JUDGE_NAME` above is defined once and reused everywhere below, rather than
> typed out twice.

In [ ]:
# ============ THE BASE JUDGE ============
from mlflow.genai.judges import make_judge

support_quality_judge = make_judge(
    name=JUDGE_NAME,
    instructions=(
        "You are rating a telecom customer-support response.\n\n"
        "Customer request: {{ inputs }}\n"
        "Agent response: {{ outputs }}\n\n"
        "Rate the response 1-5 on whether it would satisfy a real customer:\n"
        " 1: Wrong, or invents policy the company does not have.\n"
        " 2: Technically accurate but unhelpful -- the customer must ask again.\n"
        " 3: Answers the question adequately.\n"
        " 4: Answers well, anticipates the obvious follow-up.\n"
        " 5: Fully resolves the need, or correctly escalates/refuses with a clear reason.\n\n"
        "Respond with a single number from 1 to 5."
    ),
    feedback_value_type=float,
)

registered_judge = support_quality_judge.register(experiment_id=EXPERIMENT_ID)
print(f"registered base judge: {registered_judge.name}")


## Step 2 — Produce traces for the experts to rate

Use a realistic mix, not just easy cases. Experts disagree with judges most on the hard
ones — the out-of-scope questions, the rambling multi-part message, the dispute — so a
labelling set of only happy-path rows teaches alignment nothing.

Tag only the traces that were **successfully scored**, so experts don't spend time on rows
where the judge errored.

In [ ]:
# ============ GENERATE AND SCORE ============
LABEL_SET = [
    {"inputs": {"query": q, "customer_id": cid}}
    for q, cid, _, _ in T.TRAFFIC_MIX
]
print(f"{len(LABEL_SET)} distinct requests, spanning {len({n for *_, n in T.TRAFFIC_MIX})} kinds")

with mlflow.start_run(run_name="alignment_labelling_pass"):
    judge_results = mlflow.genai.evaluate(
        data=LABEL_SET,
        predict_fn=agent.answer,
        scorers=[support_quality_judge],
    )

print(f"\nunaligned judge metrics: {judge_results.metrics}")


In [ ]:
# ============ TAG THE SUCCESSFULLY-SCORED TRACES ============
result_df = judge_results.result_df
ok_trace_ids = result_df.loc[result_df["state"] == "OK", "trace_id"]

for trace_id in ok_trace_ids:
    mlflow.set_trace_tag(trace_id=trace_id, key="eval", value="complete")

print(f"tagged {len(ok_trace_ids)} traces for expert review")
print("Rows where the agent or the judge errored are excluded -- expert time is the")
print("scarcest input in this whole phase, so don't spend it on broken rows.")


## Step 3 — Build the labelling session

Three pieces: a dataset holding the traces, a **label schema** describing what to rate, and
a session assigning it to people.

Two details in the schema that matter more than they look:

- `input=InputNumeric(min_value=1, max_value=5)` must match the judge's scale, or the
  ratings aren't comparable.
- `enable_comment=True` is **not optional in practice.** MemAlign distils expert *reasoning*
  into guidelines. A bare number tells it the judge was wrong; a comment tells it why, which
  is what actually becomes a guideline.

In [ ]:
# ============ DATASET + LABEL SCHEMA + SESSION ============
from mlflow.genai import create_labeling_session, label_schemas
from mlflow.genai.datasets import create_dataset, get_dataset

try:
    label_dataset = get_dataset(name=DATASET_NAME)
    print(f"using existing dataset '{DATASET_NAME}'")
except Exception:
    label_dataset = create_dataset(name=DATASET_NAME, experiment_id=[EXPERIMENT_ID])
    print(f"created dataset '{DATASET_NAME}'")

tagged = mlflow.search_traces(
    filter_string="tags.eval = 'complete'",
    max_results=200,
    return_type="list",
)
label_dataset = label_dataset.merge_records(tagged)
print(f"dataset now holds {len(label_dataset.to_df())} records")

# The schema name MUST equal the judge name -- this is the pairing key for align().
feedback_schema = label_schemas.create_label_schema(
    name=JUDGE_NAME,
    type="feedback",
    title="Support quality (1-5)",
    input=label_schemas.InputNumeric(min_value=1.0, max_value=5.0),
    instruction=(
        "Rate this response 1-5 on whether it would satisfy a real customer.\n"
        " 1: Wrong, or invents policy the company does not have.\n"
        " 2: Technically accurate but unhelpful -- the customer must ask again.\n"
        " 3: Answers the question adequately.\n"
        " 4: Answers well, anticipates the obvious follow-up.\n"
        " 5: Fully resolves the need, or correctly escalates/refuses with a clear reason.\n\n"
        "Please leave a comment explaining any rating of 1, 2 or 5 -- the reasoning is what "
        "the alignment process actually learns from."
    ),
    enable_comment=True,
    overwrite=True,
)

session = create_labeling_session(
    name="telcoassist_support_quality_sme",
    assigned_users=SME_USERS,
    label_schemas=[JUDGE_NAME],
)
session = session.add_dataset(dataset_name=DATASET_NAME)

print()
print("Send this to your reviewers:")
print(f"  {session.url}")


### Pause here

Everything below needs completed labels. Rate the responses in the Review App yourself, or
wait for your reviewers.

**Rate honestly, especially harshly.** Alignment learns the standard you demonstrate. If you
give 4s to responses you'd actually escalate to a human, you'll get a judge that agrees with
a standard you don't hold — an expensively-produced version of the problem you started
with.

## Step 4 — Align

MemAlign reads the labelled traces, distils the pattern in the disagreements into explicit
guidelines, and writes them into the judge's instructions.

Two settings worth understanding rather than copying:

- **`embedding_model`** — set it explicitly. MemAlign embeds every trace for retrieval, and
  the default (`openai/text-embedding-3-small`) sends that off-platform and bills
  separately. A Databricks-hosted model keeps it local and cheaper.
- **`retrieval_k`** — how many similar labelled examples the judge consults per scoring
  call. Higher is more grounded and more expensive per call.

In [ ]:
# ============ RUN MEMALIGN ============
from mlflow.genai.judges.optimizers import MemAlignOptimizer
from mlflow.genai.scorers import get_scorer

traces_for_alignment = mlflow.search_traces(
    filter_string="tags.eval = 'complete'",
    max_results=200,
    return_type="list",          # align() requires the list form, not a DataFrame
)
print(f"aligning on {len(traces_for_alignment)} labelled traces")

optimizer = MemAlignOptimizer(
    reflection_lm="databricks:/databricks-claude-opus-4-6",
    retrieval_k=5,
    embedding_model="databricks:/databricks-gte-large-en",   # keep embedding cost on-platform
)

base_judge = get_scorer(name=JUDGE_NAME, experiment_id=EXPERIMENT_ID)
aligned_judge = base_judge.align(traces=traces_for_alignment, optimizer=optimizer)

print("\nGuidelines distilled from expert feedback:")
for i, guideline in enumerate(aligned_judge._semantic_memory, 1):
    print(f"  {i}. {guideline.guideline_text}")
    if guideline.source_trace_ids:
        print(f"     (from {len(guideline.source_trace_ids)} trace(s))")


Those guidelines are the deliverable. They are your experts' tacit standard written down —
often things nobody thought to put in the original judge instructions because they seemed
too obvious to state.

> **Inspecting a judge later:** `get_scorer()` returns a judge whose `_episodic_memory` looks
> **empty**, because it loads lazily on first use. That is not a failed alignment. Read
> `.instructions` instead, which contains the distilled guidelines.

In [ ]:
# ============ PERSIST THE ALIGNED JUDGE ============
from mlflow.genai.scorers import ScorerSamplingConfig

aligned_registered = aligned_judge.update(
    experiment_id=EXPERIMENT_ID,
    sampling_config=ScorerSamplingConfig(sample_rate=0.0),   # not monitoring yet
)
print(f"updated judge record: {aligned_registered.name}")

# Verify what a later session would retrieve -- .instructions, not ._episodic_memory.
retrieved = get_scorer(name=JUDGE_NAME, experiment_id=EXPERIMENT_ID)
print()
print("instructions now carry the distilled guidelines:")
print(retrieved.instructions[:700])


## Step 5 — Re-score the same responses

Critically: **the agent has not changed.** Same prompt, same knowledge base, same code. Only
the judge changed. So any difference in scores is a change in measurement, not in quality —
which is exactly what makes this the clean way to see what alignment did.

In [ ]:
# ============ SAME DATA, ALIGNED JUDGE ============
with mlflow.start_run(run_name="aligned_judge_rescore"):
    aligned_results = mlflow.genai.evaluate(
        data=LABEL_SET,
        predict_fn=agent.answer,
        scorers=[retrieved],
    )

print("unaligned:", judge_results.metrics)
print("aligned  :", aligned_results.metrics)
print()
print("If the aligned mean is lower, that is the expected outcome -- see below.")


## Step 6 — Read it correctly

This is the part to get right, and the part `GOTCHAS.md` warns about.

Compare each judge against the **human ratings**, not against each other. The cells below
use illustrative numbers so the arithmetic is visible; substitute your real expert ratings
and both judges' scores to get your own figures.

In [ ]:
# ============ AGREEMENT, BEFORE AND AFTER ============
# Replace these with your real values:
#   human_ratings  -- what the experts gave, in trace order
#   before_scores  -- the unaligned judge's scores on the same traces
#   after_scores   -- the aligned judge's scores on the same traces
human_ratings = [2, 3, 2, 4, 3, 1, 4, 3, 2, 5, 3, 2]
before_scores = [4, 5, 4, 5, 4, 3, 5, 4, 4, 5, 5, 4]
after_scores  = [2, 3, 3, 4, 3, 1, 4, 3, 2, 5, 3, 2]

report = A.alignment_report(human_ratings, before_scores, after_scores)
print(A.format_alignment_report(report))


Read that table from the bottom up. **Mean score fell by 1.4 points while every
agreement measure rose** — exact agreement from 0.08 to 0.92, weighted kappa from 0.29 to
0.96. The agent was identical in both runs.

The unaligned judge was rating this agent 4.3/5 while the people who actually understand
the domain were rating it 2.8/5. Every decision made on that 4.3 — every green gate in
Phase 5, every quiet dashboard in Phase 6 — was made on a number that was wrong by a point
and a half.

**That's the argument for this phase.** Not that alignment improves the agent. It doesn't
touch the agent. It makes every other number in the track mean something.

## Step 7 — Put the aligned judge to work

An aligned judge isn't a one-off artefact; it replaces the unaligned one everywhere:

```python
# Phase 5 -- regression gating on a standard your experts actually hold
with mlflow.start_run(run_name="candidate_v2"):
    evaluate(data=EVAL_DATASET, predict_fn=candidate, scorers=[aligned_judge])

# Phase 6 -- production monitoring, sampled
aligned_judge.start(sampling_config=ScorerSamplingConfig(sample_rate=0.10))

# Phase 8 -- the reward signal for automated prompt optimisation
optimize_prompts(..., scorers=[aligned_judge])
```

The third is Phase 8, and the dependency runs one way: GEPA optimises the agent against
whatever the judge rewards. Optimising against an unaligned judge means systematically
tuning the agent toward a standard nobody holds — which is worse than not optimising,
because it's confidently wrong.

## Key takeaways

- **Every judge in Phases 2-6 was unvalidated.** Alignment is what turns those numbers from
  plausible into trustworthy.
- **Measure agreement, not average score.** Average measures generosity; a judge that says
  5/5 to everything scores wonderfully and is useless.
- **A falling score after alignment is the success case.** The agent doesn't change; the
  measurement stops inflating. Expect the drop and read the agreement columns.
- **Use quadratic weighted kappa on ordinal scales.** Exact agreement flatters on skewed
  distributions, and plain kappa cannot distinguish "one point generous" from "random" —
  demonstrated above with identical scores for both.
- **The label schema name must equal the judge name.** Mismatch it and alignment silently
  learns nothing: no error, no pairs, no change.
- **Turn on comments.** MemAlign distils *reasoning* into guidelines; a bare number says the
  judge was wrong without saying why.
- **Set `embedding_model` explicitly**, or alignment quietly bills against an off-platform
  default for every trace it embeds.
- **After `get_scorer()`, read `.instructions`, not `._episodic_memory`** — the latter loads
  lazily and looks empty even on a perfectly aligned judge.

**Next: Phase 8 — with a judge that reflects real expert standards, the loop can be closed
automatically: GEPA optimises the prompt against that judge and promotes the winner through
Phase 5's gate.**